In [1]:
!pip -q install torch transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 40.5 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

In [3]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

DATA_PATH = "/content/train.jsonl"
VAL_PATH = "/content/val.jsonl"
OUTPUT_DIR = "/content/adapters"

os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-4
MAX_SEQ_LEN = 512

In [4]:
dataset = load_dataset(
    "json",
    data_files={"train": DATA_PATH, "validation": VAL_PATH},
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 505
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 57
    })
})

In [5]:
def format_prompt(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map = "auto"
)

model.config.use_cache = False

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.gradient_checkpointing_enable()

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj",]
)

In [11]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=25,

    save_strategy="epoch",

    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    report_to="none",
    disable_tqdm=False,

    fp16=False,
    bf16=True,

    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [18]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=format_prompt,
    data_collator=collator,
    args=training_args,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/505 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/57 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/57 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.831338
50,1.361372
75,1.182578
100,1.195659
125,1.188389
150,1.137657
175,1.039251


TrainOutput(global_step=192, training_loss=1.2715366780757904, metrics={'train_runtime': 1216.0451, 'train_samples_per_second': 1.246, 'train_steps_per_second': 0.158, 'total_flos': 2520328471105536.0, 'train_loss': 1.2715366780757904})

In [19]:
from pathlib import Path

print(OUTPUT_DIR)

# Make sure OUTPUT_DIR is a Path
OUTPUT_DIR = Path(OUTPUT_DIR)

FINAL_DIR = OUTPUT_DIR / "hr_adapter"
FINAL_DIR.mkdir(exist_ok=True, parents=True)

trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Adapter weights saved to:", FINAL_DIR)


/content/adapters
Adapter weights saved to: /content/adapters/hr_adapter


In [20]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [21]:
from threading import Thread
from transformers import TextIteratorStreamer

model.eval()
model.config.use_cache = True

test_prompt = """### Instruction:
Analyze the HR scenario step by step, clearly explain the HR reasoning, and conclude.

### Input:
What are stay bonuses and when are they used?
### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = dict(
    **inputs,
    streamer=streamer,
    max_new_tokens=128,
    temperature=0.1,
    repetition_penalty=1.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

print("### Response:")
for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()

### Response:
Stay bonuses are compensation incentives for employees who remain with a company beyond their initial employment contract term (typically 12 months). They can be structured as cash payments or other forms of recognition to encourage employee loyalty and retention. When used, they typically occur at mid-contract renewal periods (e.g., after three years) to reward long-term commitment while also acknowledging that some employees may leave during this timeframe. The decision to offer them is based on factors such as organizational need, employee preference, and business strategy.

In [23]:
import shutil
import os
shutil.make_archive('tiny_llama', 'zip', '/content/adapters')

print(f"Zip file created at: {os.getcwd()}/tiny_llama.zip")

Zip file created at: /content/tiny_llama.zip
